# Electoral Modelling — Alma Vale Result

Two reusable building blocks, used for every later data-manipulation step:

1. `CrossTableMultiplier` — join a per-location table to one or more per-region rate tables, multiply matching columns.
2. `GaussianRandomizer` — randomise a table cell-by-cell (Normal, sd = fraction of value), with an optional gate that redraws out-of-range cells.

In [21]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(r"E:\Coding Site\just_for_fun")
from random_no_generation import random_normal_adv

INFO_CSV_DIR = Path(r"E:\Coding Site\just_for_fun\Info_CSV")
ELECTORAL_MODELLING_DIR = Path(r"E:\Coding Site\just_for_fun\Electoral_Modelling")

DEMOGRAPHIC_COLS = ["S", "A1", "A2", "B1", "B2", "C1", "C2", "D1", "D2", "E1", "E2", "F1", "F2"]

### Technique 1 — cross-table multiply

In [44]:
class CrossTableMultiplier:
    """
    Joins a per-location `base` table to a per-region `rate` table (join key(s) can be one or
    several columns, e.g. Sub_Region + a local-factor column) and multiplies matching value
    columns. `apply_combined` sums several rate tables' aligned contributions before multiplying
    once (e.g. a base rate + a local adjustment).
    """

    def __init__(self, value_cols=None):
        self.value_cols = list(value_cols) if value_cols is not None else list(DEMOGRAPHIC_COLS)

    @staticmethod
    def _as_list(x):
        return list(x) if isinstance(x, (list, tuple)) else [x]

    def rate_matrix(self, base_df, rate_df, join_on, rate_join_col=None, type_col=None, type_value=None, on_missing="error"):
        """Rate table's value_cols aligned to base_df's row order (no multiply).
        on_missing: "error" raises on an unmatched key, "zero" fills it with 0."""
        join_cols = self._as_list(join_on)
        rate_join_cols = self._as_list(rate_join_col) if rate_join_col is not None else join_cols

        if type_col is not None and type_value is not None:
            rate_df = rate_df[rate_df[type_col] == type_value]

        merged = base_df[join_cols].merge(
            rate_df[rate_join_cols + self.value_cols],
            left_on=join_cols, right_on=rate_join_cols, how="left", indicator=True,
        )
        missing = merged.loc[merged["_merge"] == "left_only", join_cols].drop_duplicates()
        if len(missing):
            if on_missing == "zero":
                merged[self.value_cols] = merged[self.value_cols].fillna(0)
            else:
                raise KeyError(f"No rate row for: {missing.to_dict('records')}")

        return pd.DataFrame(merged[self.value_cols].to_numpy(), columns=self.value_cols, index=base_df.index)

    def apply(self, base_df, rate_df, join_on, rate_join_col=None, type_col=None, type_value=None, divide_by=1, on_missing="error"):
        rate = self.rate_matrix(base_df, rate_df, join_on, rate_join_col, type_col, type_value, on_missing)
        product = base_df[self.value_cols].to_numpy() * rate.to_numpy() / divide_by
        return pd.DataFrame(product, columns=self.value_cols, index=base_df.index)

    def apply_and_sum(self, base_df, rate_df, join_on, rate_join_col=None,
                       type_col=None, type_value=None, divide_by=1, out_col="Total", on_missing="error"):
        matrix = self.apply(base_df, rate_df, join_on, rate_join_col, type_col, type_value, divide_by, on_missing)
        return matrix.sum(axis=1).rename(out_col)

    def apply_combined(self, base_df, rate_specs, divide_by=1):
        """rate_specs: list of dicts with keys rate_df, join_on, rate_join_col=None, type_col=None,
        type_value=None, on_missing="error". Sums each spec's aligned rate matrix, then multiplies
        by base_df once."""
        total_rate = None
        for spec in rate_specs:
            rate = self.rate_matrix(
                base_df, spec["rate_df"], spec["join_on"],
                spec.get("rate_join_col"), spec.get("type_col"), spec.get("type_value"),
                spec.get("on_missing", "error"),
            )
            total_rate = rate if total_rate is None else total_rate + rate
        product = base_df[self.value_cols].to_numpy() * total_rate.to_numpy() / divide_by
        return pd.DataFrame(product, columns=self.value_cols, index=base_df.index)

### Technique 2 — Gaussian randomisation with an accept/redo gate

In [45]:
class GaussianRandomizer:
    """
    Draws Normal(mean=cell, sd=sd_fraction*cell) per cell via random_no_generation.random_normal_adv
    (mean == 0 cells are always left at 0 - no electorate, no draw).

    Gate (optional, low/high): redraws only cells outside [low, high] - checked against the raw
    value, or against value/reference if `reference` is passed to randomize() (e.g. gate on
    turnout% = votes / electorate) - up to max_attempts times.

    clamp_to_reference (randomize()): after gating, hard-clip any still-out-of-range cell to
    [0, reference] instead of leaving it as an outlier.
    """

    def __init__(self, sd_fraction=0.1, low=None, high=None, max_attempts=10, gate_enabled=True):
        self.sd_fraction = sd_fraction
        self.low = low
        self.high = high
        self.max_attempts = max_attempts
        self.gate_enabled = gate_enabled

    def randomize(self, means_df, reference=None, sd=None, clamp_to_reference=False):
        means = means_df.to_numpy(dtype=float)
        sds = self._resolve_sd(means, sd)
        ref = reference.to_numpy(dtype=float) if reference is not None else None

        values = self._draw(means, sds)
        attempts_used, unresolved = 1, 0
        if self.gate_enabled and (self.low is not None or self.high is not None):
            values, attempts_used, unresolved = self._gate_and_redo(values, means, sds, ref)

        if clamp_to_reference and ref is not None:
            values = np.clip(values, 0, ref)

        result = pd.DataFrame(values, columns=means_df.columns, index=means_df.index)
        return result, {"attempts_used": attempts_used, "unresolved_cells": unresolved}

    def _resolve_sd(self, means, sd):
        if sd is None:
            return means * self.sd_fraction
        if np.isscalar(sd):
            return np.full_like(means, float(sd))
        return np.asarray(sd, dtype=float).reshape(means.shape)

    def _draw(self, means, sds):
        means = np.asarray(means, dtype=float)
        sds = np.asarray(sds, dtype=float)
        flat = random_normal_adv(means.flatten().tolist(), sds.flatten().tolist())
        return np.array(flat, dtype=float).reshape(means.shape)

    def _out_of_bounds(self, values, ref):
        if ref is None:
            metric, active = values, np.ones(values.shape, dtype=bool)
        else:
            with np.errstate(divide="ignore", invalid="ignore"):
                metric = np.where(ref != 0, values / ref, 0.0)
            active = ref != 0  # no-electorate cells are trivially fine at 0, skip the gate

        bad = np.zeros(values.shape, dtype=bool)
        if self.low is not None:
            bad |= metric < self.low
        if self.high is not None:
            bad |= metric > self.high
        return bad & active

    def _gate_and_redo(self, values, means, sds, ref):
        for attempt in range(1, self.max_attempts + 1):
            bad = self._out_of_bounds(values, ref)
            if not bad.any():
                return values, attempt, 0
            values[bad] = self._draw(means[bad], sds[bad])
        return values, self.max_attempts, int(self._out_of_bounds(values, ref).sum())

`randomno.csv` is finite and can run out mid-run. This wrapper caches every uniform quantile actually drawn from it; once the file is exhausted it stops hitting the (now-useless) CSV and instead replays the cached sequence from the start, in the same order it was originally consumed, rather than raising `StopIteration`. It replaces `random_normal_adv` in place, so every `GaussianRandomizer` call above and below this cell picks it up automatically.

In [46]:
import random_no_generation as rng_module
import scipy.stats as stats

_uniform_history = []
_replay_cursor = 0
_source_exhausted = False


def resilient_random_normal_adv(input_mean, input_sd):
    """Drop-in replacement for random_no_generation.random_normal_adv. Caches every uniform
    quantile pulled from randomno.csv; once the file is exhausted, replays the cache
    cyclically (same order it was drawn) instead of raising."""
    global _replay_cursor, _source_exhausted

    non_zero_idx = [i for i, m in enumerate(input_mean) if m != 0]
    n_needed = len(non_zero_idx)
    result = [0] * len(input_mean)
    if n_needed == 0:
        return result

    uniforms = None
    if not _source_exhausted:
        raw_ints = rng_module.return_ran_array(n_needed)
        if raw_ints:
            # cache whatever was actually returned, even if it's short - a partial batch
            # still gives future replays something real to draw from
            _uniform_history.extend((x - 1) / (10_000_000 - 1) for x in raw_ints)
        if len(raw_ints) < n_needed:
            _source_exhausted = True  # randomno.csv is spent - stop hitting it, replay from here on
        else:
            uniforms = _uniform_history[-n_needed:]

    if uniforms is None:
        if not _uniform_history:
            raise RuntimeError("randomno.csv is exhausted and nothing has been cached yet to replay")
        uniforms = []
        for _ in range(n_needed):
            uniforms.append(_uniform_history[_replay_cursor % len(_uniform_history)])
            _replay_cursor += 1

    for idx, u in zip(non_zero_idx, uniforms):
        result[idx] = round(stats.norm.ppf(u, loc=input_mean[idx], scale=input_sd[idx]))
    return result


random_normal_adv = resilient_random_normal_adv  # GaussianRandomizer._draw resolves this name at call time

---

## `df_turnout`

Rate = `Turnout_Spoil_Data` (Type == Turnout, by Sub_Region) + `Turnout_Variable` (Type == Turnout, by Sub_Region + local factor). Turnout mean = electorate × rate / 100.

In [24]:
df_electorate = pd.read_csv(ELECTORAL_MODELLING_DIR / "csv_output" / "Electorate_Demographics.csv")
df_turnout_spoil = pd.read_csv(INFO_CSV_DIR / "Electoral_Modelling" / "Turnout_Spoil_Data.csv")
df_turnout_variable = pd.read_csv(
    INFO_CSV_DIR / "Electoral_Modelling" / "Enivronomental Variable" / "Turnout_Variable.csv"
)
df_polygon_embed = pd.read_csv(ELECTORAL_MODELLING_DIR / "csv_output" / "df_polygon_embed.csv")

df_electorate = df_electorate.merge(
    df_polygon_embed[["Shape_ID", "turnout_local_factors"]], on="Shape_ID", how="left"
)

turnout_multiplier = CrossTableMultiplier()
turnout_rate_specs = [
    {"rate_df": df_turnout_spoil, "join_on": "Sub_Region", "type_col": "Type", "type_value": "Turnout"},
    {
        "rate_df": df_turnout_variable,
        "join_on": ["Sub_Region", "turnout_local_factors"],
        "rate_join_col": ["Sub_Region", "Variable"],
        "type_col": "Type",
        "type_value": "Turnout",
    },
]
turnout_means = turnout_multiplier.apply_combined(df_electorate, turnout_rate_specs, divide_by=100)
turnout_means.head()

,S,A1,A2,B1,B2,C1,C2,D1,D2,E1,E2,F1,F2
0,0.00,66.30,1114.96,131.37,205.01,315.81,442.90,1279.68,259.94,407.33,1949.72,335.3,117.53
1,264.88,2298.40,574.64,1743.48,115.37,206.19,221.02,488.48,118.90,46.20,190.39,0.0,0.00
2,0.00,29.75,48.40,150.51,126.99,208.80,116.96,537.50,30.34,524.37,187.23,978.6,129.21
3,108.36,44.20,35.20,250.56,37.35,244.47,226.18,3112.34,154.98,324.94,91.64,67.2,48.18
4,0.00,38.25,536.80,73.08,119.52,201.84,259.72,602.00,152.52,263.34,736.28,168.7,85.41


Randomise: sd = 0.1 × mean, gate redraws cells where turnout% is outside [0.2, 0.95] (max 5 attempts), then clamp any still-bad cell to [0, electorate].

In [25]:
turnout_randomizer = GaussianRandomizer(sd_fraction=0.1, low=0.2, high=0.95, max_attempts=5)
turnout_values, turnout_diagnostics = turnout_randomizer.randomize(
    turnout_means, reference=df_electorate[DEMOGRAPHIC_COLS], clamp_to_reference=True
)
turnout_values = turnout_values.round().astype(int)

df_turnout = df_electorate[["County", "District", "Shape_ID", "Constituency", "Sub_Region"]].copy()
df_turnout[DEMOGRAPHIC_COLS] = turnout_values
df_turnout["Total"] = df_turnout[DEMOGRAPHIC_COLS].sum(axis=1)

print("gate diagnostics:", turnout_diagnostics)

exceeds_electorate = df_turnout[DEMOGRAPHIC_COLS].to_numpy() > df_electorate[DEMOGRAPHIC_COLS].to_numpy()
negative_turnout = df_turnout[DEMOGRAPHIC_COLS].to_numpy() < 0
bad_cells = exceeds_electorate | negative_turnout
print(f"bad cells: {exceeds_electorate.sum()} exceed electorate, {negative_turnout.sum()} negative")
if bad_cells.any():
    display(df_turnout.loc[bad_cells.any(axis=1), ["Shape_ID"] + DEMOGRAPHIC_COLS])

df_turnout.head()

E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding 

gate diagnostics: {'attempts_used': 5, 'unresolved_cells': 2901}
bad cells: 0 exceed electorate, 0 negative


,County,District,Shape_ID,Constituency,Sub_Region,S,A1,A2,B1,B2,C1,C2,D1,D2,E1,E2,F1,F2,Total
0,A01,Alington City,A01_001,The City of Alington,Capital East,0,69,985,119,200,318,438,1323,277,479,1619,310,128,6265
1,A01,Alington City,A01_002,The City of Alington,Capital East,241,1949,578,1677,117,219,220,465,132,46,186,0,0,5830
2,A01,Alington City,A01_003,The City of Alington,Capital East,0,30,50,153,106,219,121,483,30,501,172,919,145,2929
3,A01,Alington City,A01_004,The City of Alington,Capital East,115,41,30,237,38,237,235,2898,124,275,85,66,48,4429
4,A01,Alington City,A01_005,The City of Alington,Capital East,0,37,571,70,113,217,240,661,126,291,848,164,82,3420


## `df_general_spoil`

Same two-step pattern, `df_turnout` as the base instead of electorate; rate = `Turnout_Spoil_Data` (Type == Spoil_General, by Sub_Region) only - no additional/local-factor table this time.

In [26]:
spoil_multiplier = CrossTableMultiplier()
spoil_means = spoil_multiplier.apply(
    df_turnout, df_turnout_spoil, join_on="Sub_Region",
    type_col="Type", type_value="Spoil_General", divide_by=100,
)

spoil_randomizer = GaussianRandomizer(sd_fraction=0.1, low=0, high=None, max_attempts=5)
spoil_values, spoil_diagnostics = spoil_randomizer.randomize(
    spoil_means, reference=df_turnout[DEMOGRAPHIC_COLS], clamp_to_reference=True
)
spoil_values = spoil_values.round().astype(int)

df_general_spoil = df_turnout[["County", "District", "Shape_ID", "Constituency", "Sub_Region"]].copy()
df_general_spoil[DEMOGRAPHIC_COLS] = spoil_values
df_general_spoil["Total"] = df_general_spoil[DEMOGRAPHIC_COLS].sum(axis=1)

print("gate diagnostics:", spoil_diagnostics)

exceeds_turnout = df_general_spoil[DEMOGRAPHIC_COLS].to_numpy() > df_turnout[DEMOGRAPHIC_COLS].to_numpy()
negative_spoil = df_general_spoil[DEMOGRAPHIC_COLS].to_numpy() < 0
bad_cells = exceeds_turnout | negative_spoil
print(f"bad cells: {exceeds_turnout.sum()} exceed turnout, {negative_spoil.sum()} negative")
if bad_cells.any():
    display(df_general_spoil.loc[bad_cells.any(axis=1), ["Shape_ID"] + DEMOGRAPHIC_COLS])

df_general_spoil.head()

E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


gate diagnostics: {'attempts_used': 1, 'unresolved_cells': 0}
bad cells: 0 exceed turnout, 0 negative


,County,District,Shape_ID,Constituency,Sub_Region,S,A1,A2,B1,B2,C1,C2,D1,D2,E1,E2,F1,F2,Total
0,A01,Alington City,A01_001,The City of Alington,Capital East,0,0,3,0,1,1,2,6,1,3,9,3,5,34
1,A01,Alington City,A01_002,The City of Alington,Capital East,0,4,2,5,0,1,1,2,0,0,1,0,0,16
2,A01,Alington City,A01_003,The City of Alington,Capital East,0,0,0,1,1,1,1,2,0,3,1,8,5,23
3,A01,Alington City,A01_004,The City of Alington,Capital East,0,0,0,1,0,1,1,11,0,2,0,1,2,19
4,A01,Alington City,A01_005,The City of Alington,Capital East,0,0,2,0,0,1,1,3,0,2,4,1,2,16


### Verify (Constituency roll-up) — safe to delete afterwards

In [27]:
electorate_by_constituency = df_electorate.groupby("Constituency")[DEMOGRAPHIC_COLS].sum().sum(axis=1)
turnout_by_constituency = df_turnout.groupby("Constituency")[DEMOGRAPHIC_COLS].sum().sum(axis=1)
spoil_by_constituency = df_general_spoil.groupby("Constituency")[DEMOGRAPHIC_COLS].sum().sum(axis=1)

constituency_summary = pd.DataFrame({
    "Total_Electorate": electorate_by_constituency,
    "Turnout": turnout_by_constituency,
    "Spoil_Vote": spoil_by_constituency,
}).reset_index()

constituency_summary["Turnout_pct"] = (constituency_summary["Turnout"] / constituency_summary["Total_Electorate"] * 100).round(2)
constituency_summary["Spoil_pct"] = (constituency_summary["Spoil_Vote"] / constituency_summary["Turnout"] * 100).round(2)

total_row = {
    "Constituency": "Total",
    "Total_Electorate": constituency_summary["Total_Electorate"].sum(),
    "Turnout": constituency_summary["Turnout"].sum(),
    "Spoil_Vote": constituency_summary["Spoil_Vote"].sum(),
}
total_row["Turnout_pct"] = round(total_row["Turnout"] / total_row["Total_Electorate"] * 100, 2)
total_row["Spoil_pct"] = round(total_row["Spoil_Vote"] / total_row["Turnout"] * 100, 2)

constituency_summary = pd.concat([constituency_summary, pd.DataFrame([total_row])], ignore_index=True)
constituency_summary = constituency_summary[
    ["Constituency", "Total_Electorate", "Turnout", "Turnout_pct", "Spoil_Vote", "Spoil_pct"]
]
constituency_summary

,Constituency,Total_Electorate,Turnout,Turnout_pct,Spoil_Vote,Spoil_pct
0,Alington Pass,876615,722524,82.42,3297,0.46
1,Bayes Causeway,830990,645269,77.65,5909,0.92
2,Cotham Town,771266,598211,77.56,3189,0.53
3,Cyber City,1042074,847396,81.32,4512,0.53
4,Durdham Mews,713844,572688,80.23,2756,0.48
5,East Almouth,815153,633449,77.71,6353,1.00
6,East Forest,784203,581244,74.12,15109,2.60
7,East Highland,760554,614887,80.85,6264,1.02
8,Foxton Heath,1080134,877045,81.20,5204,0.59
9,Goldcrest Cradle,912134,744260,81.60,4127,0.55


## `df_referendum_spoil`

Same pattern as `df_general_spoil`: `df_turnout` as base, `Turnout_Spoil_Data` rate where Type == Spoil_Referendum.

In [73]:
referendum_spoil_multiplier = CrossTableMultiplier()
referendum_spoil_means = referendum_spoil_multiplier.apply(
    df_turnout, df_turnout_spoil, join_on="Sub_Region",
    type_col="Type", type_value="Spoil_Referendum", divide_by=100,
)

referendum_spoil_randomizer = GaussianRandomizer(sd_fraction=0.1, low=0, high=None, max_attempts=5)
referendum_spoil_values, referendum_spoil_diagnostics = referendum_spoil_randomizer.randomize(
    referendum_spoil_means, reference=df_turnout[DEMOGRAPHIC_COLS], clamp_to_reference=True
)
referendum_spoil_values = referendum_spoil_values.round().astype(int)

df_referendum_spoil = df_turnout[["County", "District", "Shape_ID", "Constituency", "Sub_Region"]].copy()
df_referendum_spoil[DEMOGRAPHIC_COLS] = referendum_spoil_values
df_referendum_spoil["Total"] = df_referendum_spoil[DEMOGRAPHIC_COLS].sum(axis=1)

print("gate diagnostics:", referendum_spoil_diagnostics)

exceeds_turnout = df_referendum_spoil[DEMOGRAPHIC_COLS].to_numpy() > df_turnout[DEMOGRAPHIC_COLS].to_numpy()
negative_spoil = df_referendum_spoil[DEMOGRAPHIC_COLS].to_numpy() < 0
bad_cells = exceeds_turnout | negative_spoil
print(f"bad cells: {exceeds_turnout.sum()} exceed turnout, {negative_spoil.sum()} negative")
if bad_cells.any():
    display(df_referendum_spoil.loc[bad_cells.any(axis=1), ["Shape_ID"] + DEMOGRAPHIC_COLS])

df_referendum_spoil.head()

gate diagnostics: {'attempts_used': 1, 'unresolved_cells': 0}
bad cells: 0 exceed turnout, 0 negative


,County,District,Shape_ID,Constituency,Sub_Region,S,A1,A2,B1,B2,C1,C2,D1,D2,E1,E2,F1,F2,Total
0,A01,Alington City,A01_001,The City of Alington,Capital East,0,0,3,0,1,1,1,3,1,1,7,3,12,33
1,A01,Alington City,A01_002,The City of Alington,Capital East,0,3,2,3,0,0,0,1,0,0,1,0,0,10
2,A01,Alington City,A01_003,The City of Alington,Capital East,0,0,0,0,0,0,0,2,0,2,1,5,8,18
3,A01,Alington City,A01_004,The City of Alington,Capital East,0,0,0,1,0,1,0,10,0,2,0,0,4,18
4,A01,Alington City,A01_005,The City of Alington,Capital East,0,0,2,0,0,0,1,2,0,1,2,1,6,15


## `df_referendum_output`

Base matrix = valid vote = `df_turnout - df_referendum_spoil`. Rate = `Election_Result_first_round` (Type == First_Round_Referendum, Variable_1 == YES); No% = 100 − Yes%. Multiply + randomise both independently, then reconcile per cell so `Turnout = Yes + No + Spoil` holds exactly: whichever of Yes/No drew the larger value keeps its draw, the other is forced to `Turnout - kept - Spoil`.

In [74]:
df_result_first_round = pd.read_csv(INFO_CSV_DIR / "Electoral_Modelling" / "Election_Result_first_round.csv")

yes_rate = df_result_first_round[
    (df_result_first_round["Type"] == "First_Round_Referendum") & (df_result_first_round["Variable_1"] == "YES")
].copy()
no_rate = yes_rate.copy()
no_rate[DEMOGRAPHIC_COLS] = 100 - no_rate[DEMOGRAPHIC_COLS]

valid_vote_df = df_turnout[["County", "District", "Shape_ID", "Constituency", "Sub_Region"]].copy()
valid_vote_df[DEMOGRAPHIC_COLS] = df_turnout[DEMOGRAPHIC_COLS] - df_referendum_spoil[DEMOGRAPHIC_COLS]

referendum_multiplier = CrossTableMultiplier()
yes_means = referendum_multiplier.apply(valid_vote_df, yes_rate, join_on="Sub_Region", divide_by=100)
no_means = referendum_multiplier.apply(valid_vote_df, no_rate, join_on="Sub_Region", divide_by=100)

yes_randomizer = GaussianRandomizer(sd_fraction=0.1, low=0, high=None, max_attempts=5)
yes_random, yes_diagnostics = yes_randomizer.randomize(
    yes_means, reference=valid_vote_df[DEMOGRAPHIC_COLS], clamp_to_reference=True
)
yes_random = yes_random.round().astype(int)

no_randomizer = GaussianRandomizer(sd_fraction=0.1, low=0, high=None, max_attempts=5)
no_random, no_diagnostics = no_randomizer.randomize(
    no_means, reference=valid_vote_df[DEMOGRAPHIC_COLS], clamp_to_reference=True
)
no_random = no_random.round().astype(int)

print("Yes gate diagnostics:", yes_diagnostics)
print("No gate diagnostics:", no_diagnostics)

Yes gate diagnostics: {'attempts_used': 1, 'unresolved_cells': 0}
No gate diagnostics: {'attempts_used': 1, 'unresolved_cells': 0}


In [77]:
turnout_arr = df_turnout[DEMOGRAPHIC_COLS].to_numpy()
spoil_arr = df_referendum_spoil[DEMOGRAPHIC_COLS].to_numpy()
yes_arr = yes_random.to_numpy()
no_arr = no_random.to_numpy()

# whichever side drew the larger value keeps its draw; the other is forced to satisfy
# Turnout = Yes + No + Spoil exactly
yes_is_max = yes_arr >= no_arr
yes_final = np.where(yes_is_max, turnout_arr - no_arr - spoil_arr, yes_arr)
no_final = np.where(yes_is_max, no_arr, turnout_arr - yes_arr - spoil_arr)

df_referendum = df_turnout[["County", "District", "Shape_ID", "Constituency", "Sub_Region"]].copy()
for i, col in enumerate(DEMOGRAPHIC_COLS):
    df_referendum[f"Yes_{col}"] = yes_final[:, i]
    df_referendum[f"No_{col}"] = no_final[:, i]

yes_cols = [f"Yes_{c}" for c in DEMOGRAPHIC_COLS]
no_cols = [f"No_{c}" for c in DEMOGRAPHIC_COLS]
df_referendum["Yes_Total"] = df_referendum[yes_cols].sum(axis=1)
df_referendum["No_Total"] = df_referendum[no_cols].sum(axis=1)

reconstructed = df_referendum[yes_cols].to_numpy() + df_referendum[no_cols].to_numpy() + spoil_arr
matches = reconstructed == turnout_arr
print(f"Yes + No + Spoil == Turnout for every cell: {bool(matches.all())} "
      f"({int((~matches).sum())} mismatches out of {matches.size})")
print(f"Grand totals -> Turnout: {int(turnout_arr.sum())}, Yes+No+Spoil: {int(reconstructed.sum())}")

df_referendum_output = df_electorate[["Shape_ID"]].copy()
df_referendum_output["Electorate"] = df_electorate[DEMOGRAPHIC_COLS].sum(axis=1)

df_referendum_output = df_referendum_output.merge(
    df_turnout[["Shape_ID", "Total"]].rename(columns={"Total": "Turnout"}), on="Shape_ID", how="left"
)
df_referendum_output = df_referendum_output.merge(
    df_referendum[["Shape_ID", "Yes_Total", "No_Total"]].rename(
        columns={"Yes_Total": "Yes_Votes", "No_Total": "No_Votes"}
    ),
    on="Shape_ID", how="left",
)
df_referendum_output = df_referendum_output.merge(
    df_referendum_spoil[["Shape_ID", "Total"]].rename(columns={"Total": "Spoil"}), on="Shape_ID", how="left"
)

for col in ["received_at", "verified_at", "declared_at"]:
    df_referendum_output[col] = ""

output_matches = (
    df_referendum_output["Yes_Votes"] + df_referendum_output["No_Votes"] + df_referendum_output["Spoil"]
    == df_referendum_output["Turnout"]
)
print(f"df_referendum_output: Yes_Votes + No_Votes + Spoil == Turnout for every row: {bool(output_matches.all())} "
      f"({int((~output_matches).sum())} mismatches out of {len(output_matches)})")

df_referendum_output.head()


Yes + No + Spoil == Turnout for every cell: True (0 mismatches out of 271050)
Grand totals -> Turnout: 25227098, Yes+No+Spoil: 25227098
df_referendum_output: Yes_Votes + No_Votes + Spoil == Turnout for every row: True (0 mismatches out of 20850)


,Shape_ID,Electorate,Turnout,Yes_Votes,No_Votes,Spoil,received_at,verified_at,declared_at
0,A01_001,8063,5697,4892,772,33,,,
1,A01_002,7316,5148,4740,398,10,,,
2,A01_003,3947,2806,2226,562,18,,,
3,A01_004,5603,4964,4288,658,18,,,
4,A01_005,3935,2967,2531,421,15,,,


## First-round score matrix (`df_first_round_score`)

Constituency-level charisma score (3 pairwise comparisons among L/MR/R, ×0.75, zero-sum) applied to every polygon in that constituency. Base score (`First_Round_General`, per Sub_Region per cluster) adjusted by swing (`First_Round_Swing_Variable`, keyed by Sub_Region + Local perception + strength; missing Strategy rows = 0). Final per party = `local_score + combined_score × (100 − total_local_score)/100`, which sums to 100 since `combined_score` (charisma + swing-adjusted base) always sums to 100 across the three parties.

In [33]:
import json

with open(ELECTORAL_MODELLING_DIR / "csv_output" / "charisma.json") as f:
    charisma = json.load(f)

sub_region_map = pd.read_csv(INFO_CSV_DIR / "sub_region_map.csv")


def charisma_pair_reward(score_a, score_b):
    """+x-y-1 to the higher score, the mirrored penalty to the lower one; 0/0 on a tie."""
    if score_a > score_b:
        r = score_a - score_b - 1
        return r, -r
    if score_a < score_b:
        r = score_b - score_a - 1
        return -r, r
    return 0, 0


constituency_lookup = sub_region_map[["Constituency", "Constituency_Code"]].drop_duplicates()

charisma_rows = []
for _, row in constituency_lookup.iterrows():
    code = row["Constituency_Code"]
    x_l = charisma["Left_Charisma"][code]
    x_mr = charisma["MR_Charisma"][code]
    x_r = charisma["Right_Charisma"][code]

    l_vs_mr, mr_vs_l = charisma_pair_reward(x_l, x_mr)
    mr_vs_r, r_vs_mr = charisma_pair_reward(x_mr, x_r)
    l_vs_r, r_vs_l = charisma_pair_reward(x_l, x_r)

    charisma_rows.append({
        "Constituency": row["Constituency"],
        "Constituency_Code": code,
        "L_charisma_score": (l_vs_mr + l_vs_r) * 0.75,
        "MR_charisma_score": (mr_vs_l + mr_vs_r) * 0.75,
        "R_charisma_score": (r_vs_mr + r_vs_l) * 0.75,
    })

constituency_charisma = pd.DataFrame(charisma_rows)

charisma_sum = (
    constituency_charisma["L_charisma_score"]
    + constituency_charisma["MR_charisma_score"]
    + constituency_charisma["R_charisma_score"]
)
print("charisma score sums to 0 for every constituency:", bool(np.allclose(charisma_sum, 0)))

constituency_charisma.head()

charisma score sums to 0 for every constituency: True


,Constituency,Constituency_Code,L_charisma_score,MR_charisma_score,R_charisma_score
0,The City of Alington,C01,0.00,0.0,0.00
1,Upper Alington,C02,1.50,-3.0,1.50
2,Alington Pass,C03,0.00,0.0,0.00
3,Durdham Mews,C04,5.25,6.0,-11.25
4,Techno Town,C05,9.00,-4.5,-4.50


In [36]:
from fractions import Fraction


def parse_fraction_or_decimal(value):
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    s = str(value).strip()
    if not s:
        return 0.0
    if "/" in s:
        return float(Fraction(s))
    return float(s)


df_swing = pd.read_csv(
    INFO_CSV_DIR / "Electoral_Modelling" / "Enivronomental Variable" / "First_Round_Swing_Variable.csv"
)
for col in DEMOGRAPHIC_COLS:
    df_swing[col] = df_swing[col].apply(parse_fraction_or_decimal)

# canonical polygon row order + all join keys needed below
polygon_keys = df_polygon_embed[["Shape_ID", "Local perception", "perception strength"]].merge(
    df_electorate[["Shape_ID", "Sub_Region", "Constituency"]], on="Shape_ID", how="left"
).rename(columns={"Local perception": "Local_perception", "perception strength": "perception_strength"})

score_multiplier = CrossTableMultiplier()
swing_rates = {}
for strategy in ["LtoM", "MtoL", "MtoR", "RtoM"]:
    strategy_table = df_swing[df_swing["Strategy"] == strategy]
    swing_rates[strategy] = score_multiplier.rate_matrix(
        polygon_keys, strategy_table,
        join_on=["Sub_Region", "Local_perception", "perception_strength"],
        rate_join_col=["Sub_Region", "Variable_1", "Variable_2"],
        on_missing="zero",
    )

swing_rates["LtoM"].head()

,S,A1,A2,B1,B2,C1,C2,D1,D2,E1,E2,F1,F2
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [39]:
general_result = df_result_first_round[df_result_first_round["Type"] == "First_Round_General"]

l_base_by_region = general_result[general_result["Variable_1"] == "L"].set_index("Sub_Region")[DEMOGRAPHIC_COLS]
mr_base_by_region = general_result[general_result["Variable_1"] == "MR"].set_index("Sub_Region")[DEMOGRAPHIC_COLS]
r_base_by_region = 100 - l_base_by_region - mr_base_by_region  # aligned by Sub_Region index

l_base_table = l_base_by_region.reset_index()
mr_base_table = mr_base_by_region.reset_index()
r_base_table = r_base_by_region.reset_index()

l_base_matrix = score_multiplier.rate_matrix(polygon_keys, l_base_table, join_on="Sub_Region")
mr_base_matrix = score_multiplier.rate_matrix(polygon_keys, mr_base_table, join_on="Sub_Region")
r_base_matrix = score_multiplier.rate_matrix(polygon_keys, r_base_table, join_on="Sub_Region")

l_party_score = l_base_matrix * (1 - swing_rates["LtoM"]) + mr_base_matrix * swing_rates["MtoL"]
mr_party_score = (
    mr_base_matrix * (1 - swing_rates["MtoL"] - swing_rates["MtoR"])
    + l_base_matrix * swing_rates["LtoM"]
    + r_base_matrix * swing_rates["RtoM"]
)
r_party_score = r_base_matrix * (1 - swing_rates["RtoM"]) + mr_base_matrix * swing_rates["MtoR"]

party_score_sum = l_party_score + mr_party_score + r_party_score
print("swing-adjusted party score sums to 100 in every cell:", bool(np.allclose(party_score_sum.to_numpy(), 100)))

swing-adjusted party score sums to 100 in every cell: True


In [41]:
polygon_charisma = polygon_keys[["Shape_ID", "Constituency"]].merge(
    constituency_charisma, on="Constituency", how="left"
)
local_scores = polygon_keys[["Shape_ID"]].merge(
    df_polygon_embed[["Shape_ID", "L_local_score", "MR_local_score", "R_local_score"]], on="Shape_ID", how="left"
)

l_char = polygon_charisma["L_charisma_score"].to_numpy()[:, None]
mr_char = polygon_charisma["MR_charisma_score"].to_numpy()[:, None]
r_char = polygon_charisma["R_charisma_score"].to_numpy()[:, None]

l_combined = l_party_score.to_numpy() + l_char
mr_combined = mr_party_score.to_numpy() + mr_char
r_combined = r_party_score.to_numpy() + r_char
total_combined = l_combined + mr_combined + r_combined

l_local = local_scores["L_local_score"].to_numpy()[:, None]
mr_local = local_scores["MR_local_score"].to_numpy()[:, None]
r_local = local_scores["R_local_score"].to_numpy()[:, None]
total_local = l_local + mr_local + r_local

print("combined score sums to 100 in every cell:", bool(np.allclose(total_combined, 100)))

weight = (100 - total_local) / 100  # local score "squeezes" combined score's weight

l_final = l_local + l_combined * weight
mr_final = mr_local + mr_combined * weight
r_final = r_local + r_combined * weight

final_sum = l_final + mr_final + r_final
print("final score sums to 100 in every cell:", bool(np.allclose(final_sum, 100)))

df_first_round_score = polygon_keys[["Shape_ID", "Constituency", "Sub_Region"]].copy()
for i, col in enumerate(DEMOGRAPHIC_COLS):
    df_first_round_score[f"L_{col}"] = l_final[:, i]
    df_first_round_score[f"MR_{col}"] = mr_final[:, i]
    df_first_round_score[f"R_{col}"] = r_final[:, i]

score_cols = [f"{p}_{c}" for p in ["L", "MR", "R"] for c in DEMOGRAPHIC_COLS]
score_values = df_first_round_score[score_cols].to_numpy()
negative_mask = score_values < 0

print(f"negative score cells: {int(negative_mask.sum())}")
if negative_mask.any():
    rows, cols = np.where(negative_mask)
    negative_scores = pd.DataFrame({
        "Shape_ID": df_first_round_score["Shape_ID"].to_numpy()[rows],
        "column": [score_cols[c] for c in cols],
        "value": score_values[rows, cols],
    })

def gate_score(score):
    return np.where(score <= 0, 1.5, np.where(score >= 100, 95, score))


gated_cells = int(((l_final <= 0) | (l_final >= 100)).sum()
                   + ((mr_final <= 0) | (mr_final >= 100)).sum()
                   + ((r_final <= 0) | (r_final >= 100)).sum())

l_gated = gate_score(l_final)
mr_gated = gate_score(mr_final)
r_gated = gate_score(r_final)

gated_sum = l_gated + mr_gated + r_gated
l_normalized = l_gated * 100 / gated_sum
mr_normalized = mr_gated * 100 / gated_sum
r_normalized = r_gated * 100 / gated_sum

for i, col in enumerate(DEMOGRAPHIC_COLS):
    df_first_round_score[f"L_{col}"] = l_normalized[:, i]
    df_first_round_score[f"MR_{col}"] = mr_normalized[:, i]
    df_first_round_score[f"R_{col}"] = r_normalized[:, i]

normalized_sum = l_normalized + mr_normalized + r_normalized
gated_values = df_first_round_score[score_cols].to_numpy()

print(f"cells gated (<=0 or >=100): {gated_cells}")
print(f"cells still <=0 or >=100 after gating: {int(((gated_values <= 0) | (gated_values >= 100)).sum())}")
print("score sums to 100 after gate + normalise:", bool(np.allclose(normalized_sum, 100)))
print(f"post-normalise value range: [{gated_values.min():.4f}, {gated_values.max():.4f}]")

df_first_round_score.head()

combined score sums to 100 in every cell: True
final score sums to 100 in every cell: True
negative score cells: 29534
cells gated (<=0 or >=100): 30638
cells still <=0 or >=100 after gating: 0
score sums to 100 after gate + normalise: True
post-normalise value range: [0.0400, 92.0000]


,Shape_ID,Constituency,Sub_Region,L_S,MR_S,R_S,L_A1,MR_A1,R_A1,L_A2,...,R_E1,L_E2,MR_E2,R_E2,L_F1,MR_F1,R_F1,L_F2,MR_F2,R_F2
0,A01_001,The City of Alington,Capital East,80.0,17.6,2.4,74.0,22.5,3.5,49.15,...,21.428571,40.313433,47.786567,11.9,31.24,34.635,34.125,20.84,23.11,56.05
1,A01_002,The City of Alington,Capital East,80.0,17.6,2.4,74.0,22.5,3.5,49.15,...,21.428571,40.313433,47.786567,11.9,31.24,34.635,34.125,20.84,23.11,56.05
2,A01_003,The City of Alington,Capital East,80.0,17.6,2.4,74.0,22.5,3.5,49.15,...,21.428571,40.313433,47.786567,11.9,31.24,34.635,34.125,20.84,23.11,56.05
3,A01_004,The City of Alington,Capital East,80.0,17.6,2.4,74.0,22.5,3.5,49.15,...,21.428571,40.313433,47.786567,11.9,31.24,34.635,34.125,20.84,23.11,56.05
4,A01_005,The City of Alington,Capital East,80.0,17.6,2.4,74.0,22.5,3.5,49.15,...,21.428571,40.313433,47.786567,11.9,31.24,34.635,34.125,20.84,23.11,56.05


## `df_general_first_round`

Base = valid vote = `df_turnout - df_general_spoil`. Rate = `df_first_round_score`'s `L_*`/`MR_*`/`R_*` (already per-Shape_ID, per-cluster, summing to 100 - a 1:1 join on `Shape_ID`, no region lookup needed). Multiply + randomise all three parties independently (params below are adjustable), then reconcile per cell to `Valid_Vote = L + MR + R` (so `Turnout = L + MR + R + Spoil`): sort the three draws ascending, keep the smallest as its own draw, cap the middle at `valid_vote - smallest` (so the remainder can't go negative), force the largest to the exact remainder. Cells with `valid_vote == 0` never cost a real draw either - `random_normal_adv` skips zero-mean cells entirely.

In [47]:
# --- adjustable parameters ---
GENERAL_SD_FRACTION = 0.2
GENERAL_GATE_LOW = 0.0
GENERAL_GATE_HIGH = 0.95
GENERAL_MAX_ATTEMPTS = 5

valid_vote_df = df_turnout[["County", "District", "Shape_ID", "Constituency", "Sub_Region"]].copy()
valid_vote_df[DEMOGRAPHIC_COLS] = df_turnout[DEMOGRAPHIC_COLS] - df_general_spoil[DEMOGRAPHIC_COLS]

l_score_table = df_first_round_score[["Shape_ID"] + [f"L_{c}" for c in DEMOGRAPHIC_COLS]].rename(
    columns={f"L_{c}": c for c in DEMOGRAPHIC_COLS}
)
mr_score_table = df_first_round_score[["Shape_ID"] + [f"MR_{c}" for c in DEMOGRAPHIC_COLS]].rename(
    columns={f"MR_{c}": c for c in DEMOGRAPHIC_COLS}
)
r_score_table = df_first_round_score[["Shape_ID"] + [f"R_{c}" for c in DEMOGRAPHIC_COLS]].rename(
    columns={f"R_{c}": c for c in DEMOGRAPHIC_COLS}
)

general_multiplier = CrossTableMultiplier()
l_means = general_multiplier.apply(valid_vote_df, l_score_table, join_on="Shape_ID", divide_by=100)
mr_means = general_multiplier.apply(valid_vote_df, mr_score_table, join_on="Shape_ID", divide_by=100)
r_means = general_multiplier.apply(valid_vote_df, r_score_table, join_on="Shape_ID", divide_by=100)

general_randomizer = GaussianRandomizer(
    sd_fraction=GENERAL_SD_FRACTION, low=GENERAL_GATE_LOW, high=GENERAL_GATE_HIGH, max_attempts=GENERAL_MAX_ATTEMPTS
)
valid_vote_ref = valid_vote_df[DEMOGRAPHIC_COLS]

l_random, l_diag = general_randomizer.randomize(l_means, reference=valid_vote_ref, clamp_to_reference=True)
mr_random, mr_diag = general_randomizer.randomize(mr_means, reference=valid_vote_ref, clamp_to_reference=True)
r_random, r_diag = general_randomizer.randomize(r_means, reference=valid_vote_ref, clamp_to_reference=True)

print("L gate diagnostics:", l_diag)
print("MR gate diagnostics:", mr_diag)
print("R gate diagnostics:", r_diag)

E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding 

L gate diagnostics: {'attempts_used': 5, 'unresolved_cells': 803}
MR gate diagnostics: {'attempts_used': 5, 'unresolved_cells': 169}
R gate diagnostics: {'attempts_used': 5, 'unresolved_cells': 21}


In [48]:
l_arr = l_random.to_numpy()
mr_arr = mr_random.to_numpy()
r_arr = r_random.to_numpy()
valid_vote_arr = valid_vote_df[DEMOGRAPHIC_COLS].to_numpy()
spoil_arr = df_general_spoil[DEMOGRAPHIC_COLS].to_numpy()

# sort the three draws ascending per cell; smallest keeps its own draw, middle is capped so
# the remainder can't go negative, largest absorbs the exact remainder
stacked = np.stack([l_arr, mr_arr, r_arr], axis=0)
order = np.argsort(stacked, axis=0)
sorted_vals = np.take_along_axis(stacked, order, axis=0)

smallest = sorted_vals[0]
middle_final = np.minimum(sorted_vals[1], valid_vote_arr - smallest)
largest_final = valid_vote_arr - smallest - middle_final

final_sorted = np.stack([smallest, middle_final, largest_final], axis=0)
final_stacked = np.empty_like(final_sorted)
np.put_along_axis(final_stacked, order, final_sorted, axis=0)
l_final_votes, mr_final_votes, r_final_votes = final_stacked

df_general_first_round = df_turnout[["County", "District", "Shape_ID", "Constituency", "Sub_Region"]].copy()
for i, col in enumerate(DEMOGRAPHIC_COLS):
    df_general_first_round[f"L_{col}"] = l_final_votes[:, i]
    df_general_first_round[f"MR_{col}"] = mr_final_votes[:, i]
    df_general_first_round[f"R_{col}"] = r_final_votes[:, i]
    df_general_first_round[f"Spoil_{col}"] = spoil_arr[:, i]

l_cols = [f"L_{c}" for c in DEMOGRAPHIC_COLS]
mr_cols = [f"MR_{c}" for c in DEMOGRAPHIC_COLS]
r_cols = [f"R_{c}" for c in DEMOGRAPHIC_COLS]
spoil_cols = [f"Spoil_{c}" for c in DEMOGRAPHIC_COLS]

df_general_first_round["L_Total"] = df_general_first_round[l_cols].sum(axis=1)
df_general_first_round["MR_Total"] = df_general_first_round[mr_cols].sum(axis=1)
df_general_first_round["R_Total"] = df_general_first_round[r_cols].sum(axis=1)
df_general_first_round["Spoil_Total"] = df_general_first_round[spoil_cols].sum(axis=1)

turnout_arr = df_turnout[DEMOGRAPHIC_COLS].to_numpy()
reconstructed = l_final_votes + mr_final_votes + r_final_votes + spoil_arr
matches = reconstructed == turnout_arr
negative_votes = (l_final_votes < 0) | (mr_final_votes < 0) | (r_final_votes < 0)

print(f"L + MR + R + Spoil == Turnout for every cell: {bool(matches.all())} "
      f"({int((~matches).sum())} mismatches out of {matches.size})")
print(f"negative vote cells: {int(negative_votes.sum())}")
print(f"Grand totals -> Turnout: {int(turnout_arr.sum())}, L+MR+R+Spoil: {int(reconstructed.sum())}")

df_general_first_round.head()

L + MR + R + Spoil == Turnout for every cell: True (0 mismatches out of 271050)
negative vote cells: 0
Grand totals -> Turnout: 25878638, L+MR+R+Spoil: 25878638


,County,District,Shape_ID,Constituency,Sub_Region,L_S,MR_S,R_S,Spoil_S,L_A1,...,R_F1,Spoil_F1,L_F2,MR_F2,R_F2,Spoil_F2,L_Total,MR_Total,R_Total,Spoil_Total
0,A01,Alington City,A01_001,The City of Alington,Capital East,0.0,0.0,0.0,0,53.0,...,98.0,3,19.0,26.0,78.0,5,2688.0,2808.0,735.0,34
1,A01,Alington City,A01_002,The City of Alington,Capital East,183.0,52.0,6.0,0,1507.0,...,0.0,0,0.0,0.0,0.0,0,3896.0,1695.0,223.0,16
2,A01,Alington City,A01_003,The City of Alington,Capital East,0.0,0.0,0.0,0,23.0,...,256.0,8,28.0,38.0,74.0,5,1250.0,1111.0,545.0,23
3,A01,Alington City,A01_004,The City of Alington,Capital East,98.0,14.0,3.0,0,28.0,...,18.0,1,10.0,12.0,24.0,2,2046.0,2003.0,361.0,19
4,A01,Alington City,A01_005,The City of Alington,Capital East,0.0,0.0,0.0,0,28.0,...,75.0,1,18.0,22.0,40.0,2,1623.0,1397.0,384.0,16


## `df_general_second_round`

Expands each party's `df_general_first_round` vote into 3 destinations (its own party's vote, split among Left/Right/MR) using `Election_Result_second_round.csv` (`Variable_1` maps to `turnout_local_factors`, same join as `Turnout_Variable`). The `Second_Round` code `Sec_Round_{pairing}_{destination}` names the pairing by the two finalists, so the *source* party is the third (excluded) one: `LM_*` -> source Right, `LR_*` -> source MR, `MR_*` -> source Left. Only 2 of each source's 3 destination rates are given; the third is `100 - other two`.

In [ ]:
df_second_round = pd.read_csv(INFO_CSV_DIR / "Electoral_Modelling" / "Election_Result_second_round.csv")
general_result_second = df_second_round[df_second_round["Type"] == "Second_Round_General"]

def pairing_table(second_round_value):
    return general_result_second[general_result_second["Second_Round"] == second_round_value][
        ["Sub_Region", "Variable_1"] + DEMOGRAPHIC_COLS
    ]

t_lm_l = pairing_table("Sec_Round_LM_L")
t_lm_m = pairing_table("Sec_Round_LM_M")
t_lr_l = pairing_table("Sec_Round_LR_L")
t_lr_r = pairing_table("Sec_Round_LR_R")
t_mr_m = pairing_table("Sec_Round_MR_M")
t_mr_r = pairing_table("Sec_Round_MR_R")

second_round_keys = df_general_first_round[["Shape_ID", "Sub_Region"]].merge(
    df_electorate[["Shape_ID", "turnout_local_factors"]], on="Shape_ID", how="left"
)

second_round_multiplier = CrossTableMultiplier()
join_kwargs = dict(join_on=["Sub_Region", "turnout_local_factors"], rate_join_col=["Sub_Region", "Variable_1"])

rate_lm_l = second_round_multiplier.rate_matrix(second_round_keys, t_lm_l, **join_kwargs)
rate_lm_m = second_round_multiplier.rate_matrix(second_round_keys, t_lm_m, **join_kwargs)
rate_lr_l = second_round_multiplier.rate_matrix(second_round_keys, t_lr_l, **join_kwargs)
rate_lr_r = second_round_multiplier.rate_matrix(second_round_keys, t_lr_r, **join_kwargs)
rate_mr_m = second_round_multiplier.rate_matrix(second_round_keys, t_mr_m, **join_kwargs)
rate_mr_r = second_round_multiplier.rate_matrix(second_round_keys, t_mr_r, **join_kwargs)

right_left_pct = rate_lm_l
right_mr_pct = rate_lm_m
right_right_pct = 100 - rate_lm_l - rate_lm_m

mr_left_pct = rate_lr_l
mr_right_pct = rate_lr_r
mr_mr_pct = 100 - rate_lr_l - rate_lr_r

left_mr_pct = rate_mr_m
left_right_pct = rate_mr_r
left_left_pct = 100 - rate_mr_m - rate_mr_r

print("Right destinations sum to 100:", bool(np.allclose((right_left_pct + right_mr_pct + right_right_pct).to_numpy(), 100)))
print("MR destinations sum to 100:", bool(np.allclose((mr_left_pct + mr_right_pct + mr_mr_pct).to_numpy(), 100)))
print("Left destinations sum to 100:", bool(np.allclose((left_mr_pct + left_right_pct + left_left_pct).to_numpy(), 100)))


In [ ]:
left_vote = df_general_first_round[[f"L_{c}" for c in DEMOGRAPHIC_COLS]].rename(columns={f"L_{c}": c for c in DEMOGRAPHIC_COLS})
mr_vote = df_general_first_round[[f"MR_{c}" for c in DEMOGRAPHIC_COLS]].rename(columns={f"MR_{c}": c for c in DEMOGRAPHIC_COLS})
right_vote = df_general_first_round[[f"R_{c}" for c in DEMOGRAPHIC_COLS]].rename(columns={f"R_{c}": c for c in DEMOGRAPHIC_COLS})

left_left_mean = left_vote * left_left_pct.to_numpy() / 100
left_right_mean = left_vote * left_right_pct.to_numpy() / 100
left_mr_mean = left_vote * left_mr_pct.to_numpy() / 100

mr_left_mean = mr_vote * mr_left_pct.to_numpy() / 100
mr_right_mean = mr_vote * mr_right_pct.to_numpy() / 100
mr_mr_mean = mr_vote * mr_mr_pct.to_numpy() / 100

right_left_mean = right_vote * right_left_pct.to_numpy() / 100
right_right_mean = right_vote * right_right_pct.to_numpy() / 100
right_mr_mean = right_vote * right_mr_pct.to_numpy() / 100

left_means_sum = left_left_mean + left_right_mean + left_mr_mean
mr_means_sum = mr_left_mean + mr_right_mean + mr_mr_mean
right_means_sum = right_left_mean + right_right_mean + right_mr_mean

print("Left means sum to Left vote:", bool(np.allclose(left_means_sum.to_numpy(), left_vote.to_numpy())))
print("MR means sum to MR vote:", bool(np.allclose(mr_means_sum.to_numpy(), mr_vote.to_numpy())))
print("Right means sum to Right vote:", bool(np.allclose(right_means_sum.to_numpy(), right_vote.to_numpy())))


In [ ]:
# --- adjustable parameters ---
SECOND_ROUND_SD_FRACTION = 0.1
SECOND_ROUND_GATE_LOW = 0.0
SECOND_ROUND_GATE_HIGH = None
SECOND_ROUND_MAX_ATTEMPTS = 5

second_round_randomizer = GaussianRandomizer(
    sd_fraction=SECOND_ROUND_SD_FRACTION, low=SECOND_ROUND_GATE_LOW,
    high=SECOND_ROUND_GATE_HIGH, max_attempts=SECOND_ROUND_MAX_ATTEMPTS,
)


def randomize_and_reconcile(source_vote_df, means_list, dest_names):
    """Randomise 3 destination means independently, then reconcile per cell so they sum
    exactly to source_vote_df: sort the 3 draws ascending, smallest keeps its own draw,
    middle is capped so the remainder can't go negative, largest absorbs the remainder."""
    source_arr = source_vote_df.to_numpy()
    draws = []
    for name, means in zip(dest_names, means_list):
        random_df, diag = second_round_randomizer.randomize(means, reference=source_vote_df, clamp_to_reference=True)
        draws.append(random_df.to_numpy())
        print(f"  {name} gate diagnostics:", diag)

    stacked = np.stack(draws, axis=0)
    order = np.argsort(stacked, axis=0)
    sorted_vals = np.take_along_axis(stacked, order, axis=0)

    smallest = sorted_vals[0]
    middle_final = np.minimum(sorted_vals[1], source_arr - smallest)
    largest_final = source_arr - smallest - middle_final

    final_sorted = np.stack([smallest, middle_final, largest_final], axis=0)
    final_stacked = np.empty_like(final_sorted)
    np.put_along_axis(final_stacked, order, final_sorted, axis=0)
    return final_stacked  # shape (3, N, 13), in dest_names order


print("Left ->")
left_final = randomize_and_reconcile(left_vote, [left_left_mean, left_right_mean, left_mr_mean], ["Left_Left", "Left_Right", "Left_MR"])
print("Right ->")
right_final = randomize_and_reconcile(right_vote, [right_left_mean, right_right_mean, right_mr_mean], ["Right_Left", "Right_Right", "Right_MR"])
print("MR ->")
mr_final = randomize_and_reconcile(mr_vote, [mr_left_mean, mr_right_mean, mr_mr_mean], ["MR_Left", "MR_Right", "MR_MR"])

df_general_second_round = df_general_first_round[["County", "District", "Shape_ID", "Constituency", "Sub_Region"]].copy()

dest_arrays = {
    "Left_Left": left_final[0], "Left_Right": left_final[1], "Left_MR": left_final[2],
    "Right_Left": right_final[0], "Right_Right": right_final[1], "Right_MR": right_final[2],
    "MR_Left": mr_final[0], "MR_Right": mr_final[1], "MR_MR": mr_final[2],
}

for name, arr in dest_arrays.items():
    for i, col in enumerate(DEMOGRAPHIC_COLS):
        df_general_second_round[f"{name}_{col}"] = arr[:, i]
    df_general_second_round[f"{name}_Total"] = arr.sum(axis=1)

left_sum = left_final[0] + left_final[1] + left_final[2]
right_sum = right_final[0] + right_final[1] + right_final[2]
mr_sum = mr_final[0] + mr_final[1] + mr_final[2]

print("Left_Left+Left_Right+Left_MR == L vote for every cell:", bool(np.array_equal(left_sum, left_vote.to_numpy())))
print("Right_Left+Right_Right+Right_MR == R vote for every cell:", bool(np.array_equal(right_sum, right_vote.to_numpy())))
print("MR_Left+MR_Right+MR_MR == MR vote for every cell:", bool(np.array_equal(mr_sum, mr_vote.to_numpy())))

grand_total_9 = sum(arr.sum() for arr in dest_arrays.values())
grand_total_lmr = left_vote.to_numpy().sum() + mr_vote.to_numpy().sum() + right_vote.to_numpy().sum()
print(f"grand total across all 9 destinations: {int(grand_total_9):,}, vs L+MR+R total: {int(grand_total_lmr):,}, equal: {grand_total_9 == grand_total_lmr}")

df_general_second_round.head()


## `df_general_output`

In [ ]:
df_general_output = df_electorate[["Shape_ID"]].copy()
df_general_output["Electorate"] = df_electorate[DEMOGRAPHIC_COLS].sum(axis=1)

df_general_output = df_general_output.merge(
    df_turnout[["Shape_ID", "Total"]].rename(columns={"Total": "Turnout"}), on="Shape_ID", how="left"
)

second_round_totals = [
    "Left_Left", "Left_Right", "Left_MR",
    "Right_Left", "Right_Right", "Right_MR",
    "MR_Left", "MR_Right", "MR_MR",
]
df_general_output = df_general_output.merge(
    df_general_second_round[["Shape_ID"] + [f"{name}_Total" for name in second_round_totals]].rename(
        columns={f"{name}_Total": name for name in second_round_totals}
    ),
    on="Shape_ID", how="left",
)

df_general_output = df_general_output.merge(
    df_general_spoil[["Shape_ID", "Total"]].rename(columns={"Total": "Spoil"}), on="Shape_ID", how="left"
)

for col in ["received_at", "verified_at", "declared_at"]:
    df_general_output[col] = ""

df_general_output.head()


In [ ]:
df_referendum_output.to_csv("./generated_result/df_results_almavale_referendum1.csv", index=False)

: 

In [79]:
df_referendum_output

,Shape_ID,Electorate,Turnout,Yes_Votes,No_Votes,Spoil,received_at,verified_at,declared_at
0,A01_001,8063,5697,4892,772,33,,,
1,A01_002,7316,5148,4740,398,10,,,
2,A01_003,3947,2806,2226,562,18,,,
3,A01_004,5603,4964,4288,658,18,,,
4,A01_005,3935,2967,2531,421,15,,,
...,...,...,...,...,...,...,...,...,...
20845,WA06_001,2159,1772,935,624,213,,,
20846,WA07_001,2210,1361,727,474,160,,,
20847,WA08_001,2104,1892,852,741,299,,,
20848,WA09_001,1763,1170,606,412,152,,,
